In [1]:
import pandas as pd
import numpy as np
from functools import reduce

In [2]:
driver = pd.read_csv('../data/driver.csv')

In [3]:
item = pd.read_csv('../data/data/coupon_item_mapping.csv') 

In [4]:
campaign = pd.read_csv('../data/data/campaign_data_clean.csv')
campaign['start_date'] = pd.to_datetime(campaign['start_date'])
campaign['end_date'] = pd.to_datetime(campaign['end_date'])
campaign.loc[campaign['start_date'] > campaign['end_date'],'end_date'] = campaign['start_date'] + pd.DateOffset(90)
campaign = campaign[['campaign_id','start_date']]
campaign = campaign.set_index('campaign_id')['start_date'].to_dict()

In [5]:
tranx = pd.read_csv('../data/data/customer_transaction_data.csv')
tranx['date'] = pd.to_datetime(tranx['date'])
tranx = tranx.merge(item, on='item_id')

In [6]:
def summary(driver, tranx, campaign, date):
    sub_driver = driver[driver['campaign_id'] == campaign]
    sub_tranx = tranx[tranx['date'] < date]
    sub_cust_coup = sub_tranx.groupby(['customer_id','coupon_id'])
    sub_cust_coup = sub_cust_coup[['quantity','selling_price','coupon_discount']].sum()
    sub_cust_coup = sub_cust_coup.reset_index()
    sub_cust_coup.columns = ['customer_id','coupon_id'] + ['cust_coup_qty','cust_coup_prc','cust_coup_cdsc']
    sub_cust = sub_tranx.groupby(['customer_id'])
    sub_cust = sub_cust[['quantity','selling_price','coupon_discount']].sum()
    sub_cust = sub_cust.reset_index()
    sub_cust.columns = ['customer_id'] + ['cust_qty','cust_prc','cust_cdsc']
    sub_coup = sub_tranx.groupby(['coupon_id'])
    sub_coup = sub_coup[['quantity','selling_price','coupon_discount']].sum()
    sub_coup = sub_coup.reset_index()
    sub_coup.columns = ['coupon_id'] + ['coup_qty','coup_prc','coup_cdsc']
    sub_driver = sub_driver.merge(sub_cust_coup, on=['customer_id','coupon_id'], how='left')
    sub_driver = sub_driver.merge(sub_cust, on=['customer_id'], how='left')
    sub_driver = sub_driver.merge(sub_coup, on=['coupon_id'], how='left')
    sub_driver = sub_driver.fillna(0)
    del sub_cust_coup, sub_cust, sub_coup, sub_tranx
    return sub_driver

In [7]:
outputs = []
for camp, date in campaign.items():
    outputs.append(summary(driver, tranx, camp, date))

In [8]:
driver = reduce(lambda x,y: x.append(y), outputs)

In [9]:
driver = driver.drop(['campaign_id','customer_id','coupon_id'], axis=1)

In [10]:
driver.shape

(128595, 10)

In [11]:
driver.head()

,id,cust_coup_qty,cust_coup_prc,cust_coup_cdsc,cust_qty,cust_prc,cust_cdsc,coup_qty,coup_prc,coup_cdsc
0,624,178.0,9169.17,-629.04,1920,149537.19,-4920.49,38365.0,2701749.09,-31576.42
1,670,52.0,10464.75,-229.39,6343,662690.08,-2181.00,38365.0,2701749.09,-31576.42
2,908,15.0,1258.80,0.00,2328,159791.20,-142.47,38365.0,2701749.09,-31576.42
3,2190,35.0,600.84,0.00,2655,178617.91,-131.08,38365.0,2701749.09,-31576.42
4,2583,88.0,4822.21,0.00,1140,112293.65,-35.62,38365.0,2701749.09,-31576.42


In [12]:
driver.to_csv('../data/feature/tranx_time_feature.csv', index=False)